# VMAE V2 网页启动
先阅读 README_TRAINING_ZH.md。将此Notebook保留在解压根目录；选择已安装PyTorch2.0.0+cu118及requirements的内核。旧检查点不在包内，默认重新训练。

In [ ]:
from pathlib import Path
import os, sys, subprocess
root = Path.cwd()
assert (root/'run_vmae_v2.py').is_file(), '请把Notebook放在解压根目录并从该目录打开'
print('Python:', sys.executable)
print('Package:', root)
subprocess.run([sys.executable, 'run_vmae_v2.py', 'check', '--gpu'], cwd=root, check=True)


## 最长序列硬件检查
先运行一次。显存不足时不要启动完整队列，应先统一调整新实验。

In [ ]:
subprocess.run([sys.executable, '-u', 'run_vmae_v2.py', 'benchmark'], cwd=root, check=True)


## 启动完整队列
后台运行，日志写入logs。不要与网页终端中的另一份任务同时启动。若只想训练15个模型，可将下面的full改为train-all。实例停机或平台清理进程会停止训练；保留results目录可恢复。

In [ ]:
logs = root/'logs'
logs.mkdir(exist_ok=True)
pidfile = logs/'notebook_job.pid'
if pidfile.exists():
    old_pid = int(pidfile.read_text())
    try:
        os.kill(old_pid, 0)
    except ProcessLookupError:
        pass
    else:
        raise RuntimeError(f'PID {old_pid} 仍存在，请先确认没有重复任务')
with (logs/'full_experiment.log').open('ab') as log:
    process = subprocess.Popen([sys.executable, '-u', 'run_vmae_v2.py', 'full'], cwd=root, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
pidfile.write_text(str(process.pid))
print('Started PID:', process.pid)


## 查看状态
此单元可重复运行。progress记录不会因任务停止而自动消失，请结合日志是否持续更新判断。

In [ ]:
subprocess.run([sys.executable, 'run_vmae_v2.py', 'status'], cwd=root, check=True)
log = root/'logs/full_experiment.log'
if log.exists():
    print('\n'.join(log.read_text(errors='replace').splitlines()[-20:]))
